# Práctica · Spark DataFrames · Clientes y pedidos

Completa las celdas indicadas. Trabaja sobre el entorno dockerizado incluido en `spark_jupyter/`.

In [1]:
from iniciar_spark import get_spark
from pyspark.sql import functions as F

spark = get_spark("PracticaClientesPedidos")
print("SparkSession creada")
print("Versión de Spark:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/18 10:42:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession creada
Versión de Spark: 4.1.1


In [2]:
from pathlib import Path

data_dir = Path("/opt/spark-apps/datos")
clientes_path = data_dir / "clientes.csv"
pedidos_path = data_dir / "pedidos.csv"

print("Ruta clientes:", clientes_path)
print("Ruta pedidos:", pedidos_path)

Ruta clientes: /opt/spark-apps/datos/clientes.csv
Ruta pedidos: /opt/spark-apps/datos/pedidos.csv


## 1. Lee ambos CSV y muestra su esquema y algunas filas.

In [3]:
clientes = spark.read.csv(
    str(clientes_path), sep=";", header=True, inferSchema=True
)
pedidos = spark.read.csv(
    str(pedidos_path), sep=";", header=True, inferSchema=True
)

print("=== Esquema de clientes ===")
clientes.printSchema()

print("=== Esquema de pedidos ===")
pedidos.printSchema()

print("=== Primeras filas de clientes ===")
clientes.show(5)

print("=== Primeras filas de pedidos ===")
pedidos.show(5)

print(f"Total clientes (con duplicados): {clientes.count()}")
print(f"Total pedidos: {pedidos.count()}")

=== Esquema de clientes ===
root
 |-- id_cliente: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- segmento: string (nullable = true)

=== Esquema de pedidos ===
root
 |-- id_pedido: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: double (nullable = true)
 |-- precio_unitario: integer (nullable = true)

=== Primeras filas de clientes ===
+----------+------+----------+--------+
|id_cliente|nombre|    ciudad|segmento|
+----------+------+----------+--------+
|         1|   Ana|  Sevilla |Estandar|
|         2|  Luis|    Bilbao| Premium|
|         3| Marta| Alicante |Estandar|
|         4| Pablo|   Madrid | Premium|
|         5| Lucia|    Bilbao| Premium|
+----------+------+----------+--------+
only showing top 5 rows
=== Primeras filas de pedidos ===
+---------+----------+----------+---------+--------+---------------+
|id_

## 2. Limpia el DataFrame de clientes: trim en ciudad y dropDuplicates.

In [4]:
# Trim en columnas de texto y eliminación de duplicados
clientes_clean = (
    clientes
    .withColumn("nombre", F.trim(F.col("nombre")))
    .withColumn("ciudad", F.trim(F.col("ciudad")))
    .withColumn("segmento", F.trim(F.col("segmento")))
    .dropDuplicates()
)

print(f"Clientes antes de limpiar: {clientes.count()}")
print(f"Clientes después de limpiar: {clientes_clean.count()}")

# Verificar valores nulos
print("=== Valores nulos por columna ===")
clientes_clean.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in clientes_clean.columns]
).show()

print("=== Clientes limpios ===")
clientes_clean.show(40)

Clientes antes de limpiar: 43
Clientes después de limpiar: 40
=== Valores nulos por columna ===
+----------+------+------+--------+
|id_cliente|nombre|ciudad|segmento|
+----------+------+------+--------+
|         0|     0|     0|       0|
+----------+------+------+--------+

=== Clientes limpios ===
+----------+--------+----------+--------+
|id_cliente|  nombre|    ciudad|segmento|
+----------+--------+----------+--------+
|        18|   Jorge|   Sevilla| Premium|
|        40|   Tomas|  Valencia|Estandar|
|         5|   Lucia|    Bilbao| Premium|
|         8|  Javier|    Madrid| Premium|
|        17|    Alba|    Madrid|Estandar|
|         2|    Luis|    Bilbao| Premium|
|        33|Patricia|  Alicante| Premium|
|        27|   Laura|    Bilbao|Estandar|
|        32|  Andres|  Zaragoza| Premium|
|        21|  Noelia|    Malaga| Premium|
|        39|     Eva|  Zaragoza|Estandar|
|        12|  Miguel|  Zaragoza|Estandar|
|        24|  Hector|  Alicante|Estandar|
|        22|   Ruben|   Gr

## 3. Convierte tipos en pedidos y crea la columna `importe`.

In [5]:
# Corrección de tipos y creación de importe = cantidad × precio_unitario
pedidos_typed = (
    pedidos
    .withColumn("cantidad", F.col("cantidad").cast("double"))
    .withColumn("precio_unitario", F.col("precio_unitario").cast("double"))
    .withColumn("fecha", F.col("fecha").cast("date"))
)

# Gestión de nulos: mostrar filas afectadas
print("=== Pedidos con cantidad nula ===")
pedidos_typed.filter(F.col("cantidad").isNull()).show()

print("=== Pedidos con producto nulo ===")
pedidos_typed.filter(F.col("producto").isNull()).show()

# Eliminar filas donde cantidad es nula (no se puede calcular importe)
pedidos_clean = (
    pedidos_typed
    .dropna(subset=["cantidad"])
    .withColumn("importe", F.round(F.col("cantidad") * F.col("precio_unitario"), 2))
)

print("=== Esquema de pedidos limpio ===")
pedidos_clean.printSchema()

print("=== Pedidos con importe calculado ===")
pedidos_clean.show(10)
print(f"Total pedidos limpios: {pedidos_clean.count()}")

=== Pedidos con cantidad nula ===
+---------+----------+----------+---------+--------+---------------+
|id_pedido|id_cliente|     fecha| producto|cantidad|precio_unitario|
+---------+----------+----------+---------+--------+---------------+
|     1006|         5|2025-03-05|  Teclado|    NULL|           35.0|
|     1028|        19|2025-03-08|  Monitor|    NULL|          226.0|
|     1084|        26|2025-03-08|Disco SSD|    NULL|          108.0|
+---------+----------+----------+---------+--------+---------------+

=== Pedidos con producto nulo ===
+---------+----------+----------+--------+--------+---------------+
|id_pedido|id_cliente|     fecha|producto|cantidad|precio_unitario|
+---------+----------+----------+--------+--------+---------------+
|     1012|        48|2025-03-05|    NULL|     1.0|          248.0|
|     1057|         7|2025-02-14|    NULL|     3.0|           25.0|
+---------+----------+----------+--------+--------+---------------+

=== Esquema de pedidos limpio ===
root


## 4. Haz un join entre clientes y pedidos.

In [6]:
# Inner join: solo clientes con pedidos y pedidos con cliente existente
ventas = clientes_clean.join(pedidos_clean, on="id_cliente", how="inner")

print(f"Clientes limpios:        {clientes_clean.count()}")
print(f"Pedidos limpios:         {pedidos_clean.count()}")
print(f"Registros tras inner join: {ventas.count()}")

# Pedidos sin cliente asociado (se pierden en el inner join)
print("\n=== Pedidos que se pierden (id_cliente no existe en clientes) ===")
pedidos_sin_cliente = pedidos_clean.join(clientes_clean, on="id_cliente", how="left_anti")
pedidos_sin_cliente.show()
print(f"Pedidos perdidos: {pedidos_sin_cliente.count()}")

# Clientes sin ningún pedido
print("\n=== Clientes sin pedidos ===")
clientes_sin_pedido = clientes_clean.join(pedidos_clean, on="id_cliente", how="left_anti")
clientes_sin_pedido.show()

print("\n=== Resultado del join ===")
ventas.show(10)

Clientes limpios:        40
Pedidos limpios:         117
Registros tras inner join: 107

=== Pedidos que se pierden (id_cliente no existe en clientes) ===
+----------+---------+----------+-----------+--------+---------------+-------+
|id_cliente|id_pedido|     fecha|   producto|cantidad|precio_unitario|importe|
+----------+---------+----------+-----------+--------+---------------+-------+
|        41|     1005|2025-03-10|    Teclado|     5.0|           40.0|  200.0|
|        48|     1012|2025-03-05|       NULL|     1.0|          248.0|  248.0|
|        46|     1022|2025-02-27|    Teclado|     3.0|           45.0|  135.0|
|        41|     1041|2025-02-23|    Teclado|     6.0|           18.0|  108.0|
|        48|     1058|2025-03-17|Auriculares|     3.0|           34.0|  102.0|
|        45|     1060|2025-03-11|  Impresora|     4.0|          193.0|  772.0|
|        42|     1074|2025-02-16|  Impresora|     2.0|          115.0|  230.0|
|        47|     1107|2025-02-12|    Monitor|     4.0| 

## 5. Filtra ventas Premium con importe >= 100.

In [7]:
ventas_premium = ventas.filter(
    (F.col("segmento") == "Premium") & (F.col("importe") >= 100)
)

print(f"Ventas Premium con importe >= 100: {ventas_premium.count()}")
ventas_premium.select(
    "id_pedido", "nombre", "ciudad", "segmento", "producto", "cantidad", "precio_unitario", "importe"
).orderBy(F.desc("importe")).show(20)

Ventas Premium con importe >= 100: 34
+---------+--------+--------+--------+-----------+--------+---------------+-------+
|id_pedido|  nombre|  ciudad|segmento|   producto|cantidad|precio_unitario|importe|
+---------+--------+--------+--------+-----------+--------+---------------+-------+
|     1117|   Ruben| Granada| Premium|   Portátil|     6.0|         1031.0| 6186.0|
|     1013|   Oscar|Alicante| Premium|   Portátil|     5.0|          741.0| 3705.0|
|     1092|    Luis|  Bilbao| Premium|   Portátil|     3.0|         1104.0| 3312.0|
|     1055|   Oscar|Alicante| Premium|   Portátil|     2.0|          776.0| 1552.0|
|     1079|   Oscar|Alicante| Premium|  Impresora|     6.0|          191.0| 1146.0|
|     1062|    Luis|  Bilbao| Premium|  Impresora|     5.0|          203.0| 1015.0|
|     1053|    Sara|  Murcia| Premium|   Portátil|     1.0|          983.0|  983.0|
|     1104|   Pablo|  Madrid| Premium|    Monitor|     5.0|          174.0|  870.0|
|     1004|   Irene|  Madrid| Premium|

## 6. Clasifica pedidos con `when` en Alto / Medio / Bajo.

In [8]:
# Alto: importe >= 500  |  Medio: 100 <= importe < 500  |  Bajo: importe < 100
ventas_clasificadas = ventas.withColumn(
    "nivel_importe",
    F.when(F.col("importe") >= 500, "Alto")
     .when(F.col("importe") >= 100, "Medio")
     .otherwise("Bajo")
)

print("=== Distribución por nivel de importe ===")
ventas_clasificadas.groupBy("nivel_importe").count().orderBy("nivel_importe").show()

print("=== Muestra con clasificación ===")
ventas_clasificadas.select(
    "id_pedido", "nombre", "producto", "importe", "nivel_importe"
).orderBy(F.desc("importe")).show(15)

=== Distribución por nivel de importe ===
+-------------+-----+
|nivel_importe|count|
+-------------+-----+
|         Alto|   32|
|         Bajo|   24|
|        Medio|   51|
+-------------+-----+

=== Muestra con clasificación ===
+---------+------+---------+-------+-------------+
|id_pedido|nombre| producto|importe|nivel_importe|
+---------+------+---------+-------+-------------+
|     1117| Ruben| Portátil| 6186.0|         Alto|
|     1061|Carlos| Portátil| 4304.0|         Alto|
|     1013| Oscar| Portátil| 3705.0|         Alto|
|     1091| David| Portátil| 3417.0|         Alto|
|     1092|  Luis| Portátil| 3312.0|         Alto|
|     1075| Marta| Portátil| 2649.0|         Alto|
|     1031| David| Portátil| 1886.0|         Alto|
|     1106|   Eva| Portátil| 1708.0|         Alto|
|     1055| Oscar| Portátil| 1552.0|         Alto|
|     1083|  Alba|Impresora| 1254.0|         Alto|
|     1079| Oscar|Impresora| 1146.0|         Alto|
|     1032| David| Portátil| 1098.0|         Alto|
|   

## 7. Calcula agregaciones por ciudad y segmento.

In [9]:
print("=== Agregaciones por ciudad ===")
ventas_clasificadas.groupBy("ciudad").agg(
    F.count("id_pedido").alias("num_pedidos"),
    F.round(F.sum("importe"), 2).alias("ingreso_total"),
    F.round(F.avg("importe"), 2).alias("importe_medio")
).orderBy(F.desc("ingreso_total")).show()

print("=== Agregaciones por segmento ===")
ventas_clasificadas.groupBy("segmento").agg(
    F.count("id_pedido").alias("num_pedidos"),
    F.round(F.sum("importe"), 2).alias("ingreso_total"),
    F.round(F.avg("importe"), 2).alias("importe_medio")
).orderBy(F.desc("ingreso_total")).show()

print("=== Agregaciones por ciudad y segmento ===")
ventas_clasificadas.groupBy("ciudad", "segmento").agg(
    F.count("id_pedido").alias("num_pedidos"),
    F.round(F.sum("importe"), 2).alias("ingreso_total"),
    F.round(F.avg("importe"), 2).alias("importe_medio")
).orderBy("ciudad", "segmento").show(40)

=== Agregaciones por ciudad ===
+----------+-----------+-------------+-------------+
|    ciudad|num_pedidos|ingreso_total|importe_medio|
+----------+-----------+-------------+-------------+
|    Bilbao|         26|      17359.0|       667.65|
|  Alicante|         17|      16585.0|       975.59|
|   Granada|         12|      10289.0|       857.42|
|    Madrid|         10|       5066.0|        506.6|
|   Sevilla|         10|       3565.0|        356.5|
|    Murcia|         11|       3448.0|       313.45|
|  Zaragoza|         10|       3243.0|        324.3|
|  Valencia|          9|       2169.0|        241.0|
|Valladolid|          1|         42.0|         42.0|
|    Malaga|          1|         34.0|         34.0|
+----------+-----------+-------------+-------------+

=== Agregaciones por segmento ===
+--------+-----------+-------------+-------------+
|segmento|num_pedidos|ingreso_total|importe_medio|
+--------+-----------+-------------+-------------+
|Estandar|         64|      34097.0|  

## 8. Crea una vista temporal y haz una consulta SQL.

In [10]:
ventas_clasificadas.createOrReplaceTempView("ventas")

resultado_sql = spark.sql("""
    SELECT
        ciudad,
        segmento,
        COUNT(id_pedido)          AS num_pedidos,
        ROUND(SUM(importe), 2)    AS ingreso_total,
        ROUND(AVG(importe), 2)    AS importe_medio,
        MAX(importe)              AS importe_maximo
    FROM ventas
    GROUP BY ciudad, segmento
    HAVING SUM(importe) > 500
    ORDER BY ingreso_total DESC
""")

print("=== Ciudades y segmentos con ingreso total > 500 ===")
resultado_sql.show(30)

=== Ciudades y segmentos con ingreso total > 500 ===
+--------+--------+-----------+-------------+-------------+--------------+
|  ciudad|segmento|num_pedidos|ingreso_total|importe_medio|importe_maximo|
+--------+--------+-----------+-------------+-------------+--------------+
|  Bilbao|Estandar|         19|      11685.0|        615.0|        3417.0|
| Granada| Premium|          7|       9161.0|      1308.71|        6186.0|
|Alicante|Estandar|         10|       9113.0|        911.3|        4304.0|
|Alicante| Premium|          7|       7472.0|      1067.43|        3705.0|
|  Bilbao| Premium|          7|       5674.0|       810.57|        3312.0|
| Sevilla|Estandar|          6|       3091.0|       515.17|        1029.0|
|  Madrid|Estandar|          5|       2719.0|        543.8|        1254.0|
|Zaragoza|Estandar|          6|       2710.0|       451.67|        1708.0|
|  Madrid| Premium|          5|       2347.0|        469.4|         870.0|
|  Murcia|Estandar|          7|       2058.0|  

## 9. Usa `sample()` y `randomSplit()`.

In [11]:
total = ventas_clasificadas.count()
print(f"Total de registros: {total}")

# sample(): extrae una fracción aleatoria
muestra = ventas_clasificadas.sample(withReplacement=False, fraction=0.3, seed=42)
print(f"\nsample(30%) → {muestra.count()} registros")
muestra.select("id_pedido", "nombre", "importe", "nivel_importe").show(5)

# randomSplit(): divide en particiones (útil para train/test en ML)
train, test = ventas_clasificadas.randomSplit([0.8, 0.2], seed=42)
print(f"\nrandomSplit([0.8, 0.2]):")
print(f"  Train → {train.count()} registros")
print(f"  Test  → {test.count()} registros")

Total de registros: 107

sample(30%) → 32 registros
+---------+--------+-------+-------------+
|id_pedido|  nombre|importe|nivel_importe|
+---------+--------+-------+-------------+
|     1094|   Tomas|  768.0|         Alto|
|     1003|   Lucia|  420.0|        Medio|
|     1062|    Luis| 1015.0|         Alto|
|     1111|Patricia|  246.0|        Medio|
|     1063|  Andres|  201.0|        Medio|
+---------+--------+-------+-------------+
only showing top 5 rows

randomSplit([0.8, 0.2]):
  Train → 89 registros
  Test  → 18 registros


## 10. Guarda el resultado en Parquet en `/opt/spark-apps/salida/resultado_parquet` y léelo de nuevo.

In [12]:
output_path = "/opt/spark-apps/salida/resultado_parquet"

# Guardar en Parquet
resultado_sql.write.mode("overwrite").parquet(output_path)
print(f"Datos guardados en: {output_path}")

# Leer de nuevo desde Parquet
resultado_parquet = spark.read.parquet(output_path)
print("\n=== Datos leídos desde Parquet ===")
resultado_parquet.printSchema()
resultado_parquet.show(30)
print(f"Total registros leídos: {resultado_parquet.count()}")

Datos guardados en: /opt/spark-apps/salida/resultado_parquet

=== Datos leídos desde Parquet ===
root
 |-- ciudad: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- num_pedidos: long (nullable = true)
 |-- ingreso_total: double (nullable = true)
 |-- importe_medio: double (nullable = true)
 |-- importe_maximo: double (nullable = true)

+--------+--------+-----------+-------------+-------------+--------------+
|  ciudad|segmento|num_pedidos|ingreso_total|importe_medio|importe_maximo|
+--------+--------+-----------+-------------+-------------+--------------+
|  Bilbao|Estandar|         19|      11685.0|        615.0|        3417.0|
| Granada| Premium|          7|       9161.0|      1308.71|        6186.0|
|Alicante|Estandar|         10|       9113.0|        911.3|        4304.0|
|Alicante| Premium|          7|       7472.0|      1067.43|        3705.0|
|  Bilbao| Premium|          7|       5674.0|       810.57|        3312.0|
| Sevilla|Estandar|          6|       3091

## 11. Responde brevemente en Markdown:
- ¿Qué ventaja tiene usar join?
- ¿Qué diferencia hay entre sample y randomSplit?
- ¿Qué pedido se pierde en el inner join y por qué?

---

**¿Qué ventaja tiene usar join?**  
El join permite combinar información de dos fuentes distintas usando una clave común (`id_cliente`), obteniendo una visión unificada del negocio (datos del cliente + datos del pedido) sin necesidad de duplicar información en un solo fichero. Además, Spark optimiza la ejecución mediante broadcast joins o sort-merge joins según el tamaño de los DataFrames.

**¿Qué diferencia hay entre sample y randomSplit?**  
`sample(fraction)` extrae una fracción aleatoria del DataFrame original y devuelve **un solo** DataFrame más pequeño, útil para exploración o pruebas rápidas. `randomSplit([0.8, 0.2])` divide el DataFrame en **múltiples particiones disjuntas** (sin solapamiento), lo que es habitual para separar conjuntos de entrenamiento y test en proyectos de Machine Learning.

**¿Qué pedido se pierde en el inner join y por qué?**  
Se pierden los pedidos cuyo `id_cliente` no existe en la tabla de clientes: los id_cliente **41, 42, 45, 46, 47 y 48** aparecen en `pedidos.csv` pero no en `clientes.csv`. El inner join solo conserva las filas con correspondencia en ambas tablas, por lo que estos pedidos (registros "huérfanos") quedan excluidos del resultado. También se excluyen los clientes que no tienen ningún pedido asociado.

## 12. Cierra la sesión de Spark.

In [ ]:
spark.stop()